In [1]:
# Install required packages in Google Colab
# !pip install -qq weaviate-client python-dotenv weaviate-agents datasets

# Weaviate Query Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weaviate/weaviate_query_agent_demo/blob/main/query_agent_functionality.ipynb)

Here, we'll walk you through a comprehensive example showcasing the **Weaviate Query Agent** functionality.  
The Query Agent is an intelligent layer that sits on top of your Weaviate vector database, using generative AI to optimize natural language queries and automatically determine the best search strategy.

### How the Query Agent Works

The Query Agent intelligently handles complex queries by:
- **Query Optimization**: Uses generative AI to transform natural language into optimized vector database queries
- **Collection Selection**: Automatically determines which collection(s) to search based on your question
- **Smart Filtering**: Decides when filtering is needed to narrow down results
- **Aggregation Logic**: Determines if aggregation operations should be performed (counts, grouping, etc.)
- **Execution**: Runs the optimized queries against your Weaviate instance

### Dataset Overview

This notebook demonstrates the Query Agent using three diverse collections:
- **Books**: 10,000 books with titles, authors, descriptions, and genres
- **Brands**: 104 clothing brands with descriptions, ratings, and company information  
- **Ecommerce**: 448 fashion items with prices, categories, reviews, and brand associations

All datasets come pre-vectorized with **Snowflake Arctic Embed v2.0** embeddings via Weaviate's embedding service.

### Further Reading

- Check out the datasets on [HuggingFace](https://huggingface.co/datasets/weaviate/agents)
- Learn more about Query Agents in the [official documentation](https://docs.weaviate.io/agents/query)
- Explore the technical implementation in our [Query Agent tutorial](https://docs.weaviate.io/agents/query/tutorial-ecommerce)
- Understand vector databases in the [Weaviate developer docs](https://weaviate.io/developers/weaviate)

If running locally, to ensure smooth execution and prevent potential conflicts with your global Python environment, we recommend following the instructions in the README running the code in a virtual environment.

<a id='TOC'></a>  
## Table of contents  

1. <a href="#Dependencies">Dependencies</a><br>
2. <a href="#Connecting">Connecting to your Weaviate cluster</a><br>
3. <a href="#Collections">Setting up collections and loading data</a><br>
   3.1 <a href="#BrandsCollection">Create and populate Brands collection</a><br>
   3.2 <a href="#EcommerceCollection">Create and populate Ecommerce collection</a><br>
   3.3 <a href="#BooksCollection">Create and populate Books collection</a><br>
4. <a href="#QueryAgent">Creating the Query Agent</a><br>
5. <a href="#ExampleQueries">Example Query Agent interactions</a><br>
   5.1 <a href="#ClassicRAG">Classic RAG search</a><br>
   5.2 <a href="#DatabaseFiltering">Generated database filtering</a><br>
   5.3 <a href="#StatisticalQueries">Statistical and aggregation queries</a><br>
   5.4 <a href="#MultipleQueries">Querying multiple databases</a><br>
   5.5 <a href="#MissingInfo">Identifying missing information</a><br>
6. <a href="#ContractAnalysis">Practical Business Use Case for the Query Agent: Contract Analysis</a><br>
   6.1 <a href="#AgentTask1">Agent Task 1: Risk Assessment</a><br>
   6.2 <a href="#AgentTask2">Agent Task 2: Contract Comparison</a><br>
   6.3 <a href="#AgentTask3">Agent Task 3: Contract Drafting Suggestions</a><br>
   6.4 <a href="#MultiStepWorkflows">Multi-Step Agent Workflows</a><br>
   6.5 <a href="#BusinessScenarios">Business Scenarios</a><br>

<a id='Dependencies'></a>  
## 1. Dependencies
[Back to table of contents](#TOC)

This section initializes the necessary dependencies and sets environment variables for connecting to your Weaviate Cloud instance.

**If using Colab, add your keys between the quotation marks in the cell below**

In [19]:
# Run this cell to load your keys in Colab
import os

# set environment variables
os.environ['WEAVIATE_URL'] = ''  # weaviate instance url
os.environ['WEAVIATE_API_KEY'] = ''  # admin api key

**If running locally, make sure your keys are in your .env file and run this cell**

In [20]:
# Run this cell if you are working locally in VS code or Code Space
import dotenv

dotenv.load_dotenv(override=True)

True

<a id='Connecting'></a>  
## 2. Connecting to your Weaviate cluster  
[Back to table of contents](#TOC)  

To interact with our Weaviate cluster, we'll initialize a client object and verify the connection is successful. This connection will be used throughout the notebook for all data operations and Query Agent interactions.

In [21]:
import weaviate, os

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY"))
)

# check if the connection is successful
client.is_ready()

True

<a id='Collections'></a>  
## 3. Setting up collections and loading data  
[Back to table of contents](#TOC)  

In this section, we'll create three collections (Brands, Ecommerce, and Books) and populate them with pre-vectorized data from the [Weaviate Agents dataset on Hugging Face](https://huggingface.co/datasets/weaviate/agents). Each collection uses the Snowflake Arctic Embed v2.0 model for consistent vector embeddings.

<a id='BrandsCollection'></a>  
### 3.1 Create and populate Brands collection  
[Back to table of contents](#TOC)  

We'll start by creating the Brands collection with properties for clothing brand information, then populate it with brand records from the Weaviate agents dataset.

In [5]:
# import required module for configuration
import weaviate.classes.config as wc

# delete collection if it exists
if client.collections.exists("Brands"):
    client.collections.delete("Brands")

client.collections.create(
    name="Brands", #Collection name

    #Set vectorizer configuration for text2vec using Weaviates embedding model
    vector_config=wc.Configure.Vectors.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
        source_properties=["name", "description", "child_brands", "parent_brand"],
    ),

)

print("Successfully created collection: Brands.")

Successfully created collection: Brands.


In [6]:
# Upload brands data using the Weaviate-recommended streaming approach
from datasets import load_dataset

# Load fresh streaming dataset for upload
brands_dataset = load_dataset("weaviate/agents", "query-agent-brands", split="train", streaming=True)

brands_collection = client.collections.get("Brands")
with brands_collection.batch.fixed_size(batch_size=100) as batch:
    for item in brands_dataset:
        batch.add_object(
            properties=item["properties"],
            vector=item["vector"]
        )

print("Successfully uploaded brands data using dataset streaming method!")

Successfully uploaded brands data using dataset streaming method!


<a id='EcommerceCollection'></a>  
### 3.2 Create and populate Ecommerce collection  
[Back to table of contents](#TOC)  

Next, we'll create the Ecommerce collection for fashion items with properties including prices, categories, reviews, and brand associations, then load product records.

In [7]:
# import required module for configuration
import weaviate.classes.config as wc

# delete collection if it exists
if client.collections.exists("Ecommerce"):
    client.collections.delete("Ecommerce")

client.collections.create(
    name="Ecommerce", #Collection name

    #Set vectorizer configuration for text2vec using Weaviates embedding model
    vector_config=wc.Configure.Vectors.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
        source_properties=["name", "description", "collection", "category", "subcategory", "brand", "colors", "tags", "reviews"],
    ),
)

print("Successfully created collection: Ecommerce.")

Successfully created collection: Ecommerce.


In [8]:
# Upload ecommerce data using the Weaviate-recommended streaming approach
ecommerce_dataset = load_dataset("weaviate/agents", "query-agent-ecommerce", split="train", streaming=True)

ecommerce_collection = client.collections.get("Ecommerce")
with ecommerce_collection.batch.fixed_size(batch_size=100) as batch:
    for item in ecommerce_dataset:
        batch.add_object(
            properties=item["properties"],
            vector=item["vector"]
        )

print("Successfully uploaded ecommerce data using dataset streaming method!")

Successfully uploaded ecommerce data using dataset streaming method!


<a id='BooksCollection'></a>  
### 3.3 Create and populate Books collection  
[Back to table of contents](#TOC)  

Finally, we'll create the Books collection with properties for titles, authors, descriptions, and genres, then populate it with book records from various genres including mystery, fiction, and non-fiction.

In [9]:
# import required module for configuration
import weaviate.classes.config as wc

# delete collection if it exists
if client.collections.exists("Books"):
    client.collections.delete("Books")

client.collections.create(
    name="Books", #Collection name

    #Set vectorizer configuration for text2vec using Weaviates embedding model
    vector_config=wc.Configure.Vectors.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
        source_properties=["title", "author", "description", "genres"],
    ),
)

print("Successfully created collection: Books.")

Successfully created collection: Books.


In [10]:
# Upload books data using the Weaviate-recommended streaming approach
books_dataset = load_dataset("weaviate/agents", "query-agent-books", split="train", streaming=True)

books_collection = client.collections.get("Books")
with books_collection.batch.fixed_size(batch_size=100) as batch:
    for item in books_dataset:
        batch.add_object(
            properties=item["properties"],
            vector=item["vector"]
        )

print("Successfully uploaded books data using dataset streaming method!")

Successfully uploaded books data using dataset streaming method!


<a id='QueryAgent'></a>  
## 4. Creating the Query Agent  
[Back to table of contents](#TOC)  

Now we'll create our Query Agent instance, which will intelligently route queries across our three collections (Brands, Ecommerce, Books) and automatically determine the optimal search strategy.

The output from the Query Agent will include: the original query, the generated answer to the query, the searches performed, the collections searched, filters applied, aggregates completed, source objects pulled from the database that comntributed to the generated answer and any mising information if the generated answer is incomplete.

In [22]:
from weaviate.agents.query import QueryAgent

# Instantiate a new agent object, and specify the collections to query
qa = QueryAgent(
    client=client,
    collections=["Brands", "Ecommerce", "Books", "Financialcontracts"],
)


<a id='ExampleQueries'></a>  
## 5. Example Query Agent interactions  
[Back to table of contents](#TOC)  

Let's explore the Query Agent's capabilities through various types of queries. Notice how the agent automatically determines which collections to search, applies filters, and performs aggregations based on the natural language input.

<a id='BookQueries'></a>  
### 5.0 Classic Vector Retrieval 
[Back to table of contents](#TOC)  

This Query Agent is able to perform classic vector database retrieval without initiating a subsequent generative step.

In [23]:
# Perform a vector search
response = qa.search(
    "Books about plants"
)

# Print the response
# The output will include: the original query, the generated answer to the query, the seraches performed,
# the collections searched, filters applied, aggregates completed and source objects ulled from the database.
print(response)

searches=[QueryResultWithCollectionNormalized(query='books about plants, plant biology, botany, horticulture, gardening, flora, vegetation', filters=None, collection='Books')] usage=ModelUnitUsage(model_units=115, usage_in_plan=True, remaining_plan_requests=931) total_time=4.30177116394043 search_results=QueryReturn(objects=[Object(uuid=UUID('c5788159-c428-45d3-b882-f5058a0e9b22'), metadata={'creation_time': None, 'last_update_time': None, 'distance': None, 'certainty': None, 'score': 0.8203125, 'explain_score': None, 'is_consistent': None, 'rerank_score': None}, properties={'description': 'Every schoolchild learns about the mutually beneficial dance of honeybees and flowers: The bee collects nectar and pollen to make honey and, in the process, spreads the flowers’ genes far and wide. In The Botany of Desire, Michael Pollan ingeniously demonstrates how people and domesticated plants have formed a similarly reciprocal relationship. He masterfully links four fundamental human desires—swe

<a id='BookQueries'></a>  
### 5.1 Classic RAG search  
[Back to table of contents](#TOC)  

This Query Agent is able to respond as a classic RAG system. The Query Agent will run an initial generative step that will optimize the original query for a vector database.



In [27]:
# Perform a query
response = qa.ask(
    "Are there any books about King Arthur or Knights that I should read?"
)

# Print the response
# The output will include: the original query, the generated answer to the query, the seraches performed,
# the collections searched, filters applied, aggregates completed and source objects ulled from the database.
response.display()

AskModeResponse(
│   output_type='final_state',
│   searches=[
│   │   QueryResultWithCollectionNormalized(query='King Arthur', filters=None, collection='Books'),
│   │   QueryResultWithCollectionNormalized(query='books about knights', filters=None, collection='Books')
│   ],
│   aggregations=[],
│   usage=ModelUnitUsage(model_units=228, usage_in_plan=True, remaining_plan_requests=915),
│   total_time=6.994572877883911,
│   is_partial_answer=False,
│   missing_information=[],
│   final_answer='Here are some highly recommended books about King Arthur and knights you might enjoy:\n\n1. "The Winter King" by Bernard Cornwell – A historical fantasy novel about Arthur\'s struggle to unite Britain amidst chaos, featuring Merlin\'s magic and Arthur\'s romance with Guinevere.\n\n2. "The Once and Future King" by T.H. White – A classic and masterful retelling of the Arthurian legend, blending comedy and tragedy with warmth and charm.\n\n3. "A Knight of the Seven Kingdoms" by George R.R. Martin – A collection of novellas set a century before "A Game of Thrones," following the adventures of Ser Duncan the Tall, a hedge knight, and his squire Egg in a world of chivalry, royal intrigue, and adventure.\n\nThese books offer different perspectives and rich storytelling about King Arthur and knights.',
│   sources=[
│   │   Source(object_id='3033622e-aaa0-4994-95e7-6b7f4d8b8583', collection='Books'),
│   │   Source(object_id='c207796d-1477-4fd7-a1f9-2a70e144a4ae', collection='Books'),
│   │   Source(object_id='948679d7-57d7-4cc4-9c99-ed404af99bb2', collection='Books')
│   ]
)

<a id='BookQueries'></a>  
### 5.2 Generated database filtering   
[Back to table of contents](#TOC)  

This Query Agent is able determine appropriate filters and apply filtering for database objects returned by the vector database query. This allows for dynamic filtering in RAG systems, a step that is normally a manual process.


In [28]:
# Perform a query
response = qa.ask(
    "I am in the mood for a good mystery book, what should I read?"
)

# Print the response
response.display()

AskModeResponse(
│   output_type='final_state',
│   searches=[QueryResultWithCollectionNormalized(query='mystery books', filters=None, collection='Books')],
│   aggregations=[],
│   usage=ModelUnitUsage(model_units=145, usage_in_plan=True, remaining_plan_requests=911),
│   total_time=6.571720600128174,
│   is_partial_answer=False,
│   missing_information=[],
│   final_answer="If you're in the mood for a gripping mystery, here are some excellent book recommendations:\n\n1. **The Keepsake (Rizzoli & Isles, #7) by Tess Gerritsen** - This New York Times bestseller is a suspenseful thriller involving a mysterious mummy that turns out to be a modern-day murder victim. Medical examiner Maura Isles and detective Jane Rizzoli race against time to stop a killer with a dark secret tied to ancient death rituals.\n\n2. **Cross Bones (Temperance Brennan, #8) by Kathy Reichs** - Follow forensic anthropologist Temperance Brennan as she investigates a bizarre murder linked to an ancient biblical mystery surrounding the discovery of Christ's tomb. This one blends crime, mystery, and historical intrigue brilliantly.\n\n3. **The Last Book by Zoran Živković** - A literary mystery where inspector Dejan Lukić investigates a series of deaths connected to an elusive book. It’s a unique blend of mystery, thriller, and literary fiction with a clever and unexpected ending.\n\nAny of these would offer a compelling and suspenseful mystery read depending on what style you prefer—medical/crime thriller, archaeological biblical mystery, or bookish thriller. Happy reading!",
│   sources=[
│   │   Source(object_id='248210ee-45bc-4257-b762-9a43b684d629', collection='Books'),
│   │   Source(object_id='b0387495-b824-4ee0-9ff2-ff249296a2ea', collection='Books'),
│   │   Source(object_id='6b5ae9f5-52d0-4440-a8f9-06d62a7a1e2b', collection='Books')
│   ]
)

<a id='StatisticalQueries'></a>  
### 5.3 Statistical and aggregation queries  
[Back to table of contents](#TOC)  

The Query Agent automatically recognizes when aggregation is needed and performs complex statistical operations like counting and grouping data.

In [29]:
# Perform a query
response = qa.ask(
    "Which author has the most books in my collection?"
)

# Print the response
response.display()

AskModeResponse(
│   output_type='final_state',
│   searches=[],
│   aggregations=[
│   │   AggregationResultWithCollectionNormalized(
│   │   │   groupby_property='author',
│   │   │   aggregation=TextPropertyAggregation(
│   │   │   │   property_name='title',
│   │   │   │   metrics=<TextMetrics.COUNT: 'COUNT'>,
│   │   │   │   top_occurrences_limit=None
│   │   │   ),
│   │   │   filters=None,
│   │   │   collection='Books'
│   │   )
│   ],
│   usage=ModelUnitUsage(model_units=575, usage_in_plan=True, remaining_plan_requests=907),
│   total_time=5.904170989990234,
│   is_partial_answer=False,
│   missing_information=[],
│   final_answer='The author with the most books in your collection is Stephen King, with a total of 57 books.',
│   sources=[]
)

<a id='MultipleQueries'></a>  
### 5.4 Querying multiple databases  
[Back to table of contents](#TOC)  

The Query agent looks at collection descriptions and property data to determine which collection or collections to query for proper context for the query response generation.

In [30]:
# Perform a query
response = qa.ask(
    """I am looking to buy a cool new jean jacket or light coat at or below $150.
    I want a trendy but also timeless look but can also hold up and last me a long time.
    If possible, I want to purchase something from a new company that might get me a
    sponsorship for my social media account."""
)
# Print the response
response.display()

AskModeResponse(
│   output_type='final_state',
│   searches=[
│   │   QueryResultWithCollectionNormalized(query='jean jacket', filters=None, collection='Ecommerce'),
│   │   QueryResultWithCollectionNormalized(query='light coat', filters=None, collection='Ecommerce'),
│   │   QueryResultWithCollectionNormalized(
│   │   │   query='new clothing brands',
│   │   │   filters=IntegerPropertyFilter(
│   │   │   │   property_name='foundation_year',
│   │   │   │   operator=<ComparisonOperator.GREATER_EQUAL: '>='>,
│   │   │   │   value=2015.0
│   │   │   ),
│   │   │   collection='Brands'
│   │   ),
│   │   QueryResultWithCollectionNormalized(
│   │   │   query='trendy timeless jean jacket durability',
│   │   │   filters=IntegerPropertyFilter(
│   │   │   │   property_name='price',
│   │   │   │   operator=<ComparisonOperator.LESS_EQUAL: '<='>,
│   │   │   │   value=150.0
│   │   │   ),
│   │   │   collection='Ecommerce'
│   │   ),
│   │   QueryResultWithCollectionNormalized(
│   │   │   query='trendy timeless light coat durability',
│   │   │   filters=IntegerPropertyFilter(
│   │   │   │   property_name='price',
│   │   │   │   operator=<ComparisonOperator.LESS_EQUAL: '<='>,
│   │   │   │   value=150.0
│   │   │   ),
│   │   │   collection='Ecommerce'
│   │   )
│   ],
│   aggregations=[
│   │   AggregationResultWithCollectionNormalized(
│   │   │   groupby_property=None,
│   │   │   aggregation=IntegerPropertyAggregation(property_name='price', metrics=<NumericMetrics.MEAN: 'MEAN'>),
│   │   │   filters=FilterAndOr(
│   │   │   │   combine='OR',
│   │   │   │   filters=[
│   │   │   │   │   TextPropertyFilter(
│   │   │   │   │   │   property_name='name',
│   │   │   │   │   │   operator=<ComparisonOperator.LIKE: 'LIKE'>,
│   │   │   │   │   │   value='*jean jacket*'
│   │   │   │   │   ),
│   │   │   │   │   TextPropertyFilter(
│   │   │   │   │   │   property_name='name',
│   │   │   │   │   │   operator=<ComparisonOperator.LIKE: 'LIKE'>,
│   │   │   │   │   │   value='*light coat*'
│   │   │   │   │   )
│   │   │   │   ]
│   │   │   ),
│   │   │   collection='Ecommerce'
│   │   ),
│   │   AggregationResultWithCollectionNormalized(
│   │   │   groupby_property=None,
│   │   │   aggregation=IntegerPropertyAggregation(
│   │   │   │   property_name='foundation_year',
│   │   │   │   metrics=<NumericMetrics.COUNT: 'COUNT'>
│   │   │   ),
│   │   │   filters=IntegerPropertyFilter(
│   │   │   │   property_name='foundation_year',
│   │   │   │   operator=<ComparisonOperator.GREATER_EQUAL: '>='>,
│   │   │   │   value=2022.0
│   │   │   ),
│   │   │   collection='Brands'
│   │   )
│   ],
│   usage=ModelUnitUsage(model_units=683, usage_in_plan=True, remaining_plan_requests=903),
│   total_time=12.536434888839722,
│   is_partial_answer=True,
│   missing_information=[
│   │   "The answer does not specify which brands associated with the jackets and coats are new or recently founded, beyond mentioning them as 'new or innovative'. More explicit detail on brand foundation dates or newness is missing.",
│   │   'There is no direct mention from the information if any of these brands have established sponsorship or influencer partnership programs, the user was interested in sponsorship potential.'
│   ],
│   final_answer="You can find some excellent jean jackets and light coats under $150 that offer both a trendy yet timeless look and durability. Here are a few options from newer and innovative brands that may also align with your interest in potential sponsorships:\n\n1. **Garden Dusk Linen Jacket by Loom & Aura** - $90\n   - A soft linen jacket in a subtle lavender color, designed for comfort and style with thoughtful detailing.\n   - It has a charming, nature-inspired aesthetic that's both trendy and timeless.\n\n2. **Garden Twilight Quilted Jacket by Echo & Stitch** - $120\n   - Quilted for warmth and featuring delicate stitching in earth tones, perfect for cooler evenings.\n   - Known for cozy comfort and a stylish finish.\n\n3. **Rustic

<a id='MissingInfo'></a>  
### 5.5 Identifying missing information  
[Back to table of contents](#TOC)  

If a partial response is generated for a query the Query Agent will report the missing information not present in the database.

In [31]:
# Perform a query
response = qa.ask(
    """I want to buy a new leather jacket but I am vegan and only buy from socially responsible brands."""
)
# Print the response
response.display()

AskModeResponse(
│   output_type='final_state',
│   searches=[
│   │   QueryResultWithCollectionNormalized(
│   │   │   query='vegan leather jacket socially responsible brand',
│   │   │   filters=None,
│   │   │   collection='Ecommerce'
│   │   ),
│   │   QueryResultWithCollectionNormalized(
│   │   │   query='ethical vegan leather jacket',
│   │   │   filters=None,
│   │   │   collection='Ecommerce'
│   │   ),
│   │   QueryResultWithCollectionNormalized(
│   │   │   query='cruelty-free vegan leather jacket',
│   │   │   filters=None,
│   │   │   collection='Ecommerce'
│   │   ),
│   │   QueryResultWithCollectionNormalized(
│   │   │   query='leather jacket vegan socially responsible ethical practices',
│   │   │   filters=None,
│   │   │   collection='Brands'
│   │   )
│   ],
│   aggregations=[],
│   usage=ModelUnitUsage(model_units=369, usage_in_plan=True, remaining_plan_requests=899),
│   total_time=10.341312170028687,
│   is_partial_answer=True,
│   missing_information=[
│   │   'Specific information about vegan leather jackets from socially responsible brands is missing. The answer does not confirm any jacket or brand in the provided sources as vegan or socially responsible.',
│   │   "Details about the ethical practices or social responsibility commitments of brands mentioned (Vivid Verse, Loom & Aura, Velvet Vesper) are not available, making it unclear whether these brands meet the user's criteria."
│   ],
│   final_answer='For a vegan and socially responsible leather jacket, you should look for brands that specifically focus on cruelty-free, ethical, and sustainable fashion. While the results mention some jackets and brands like "Vivid Verse" and "Loom & Aura," it does not explicitly confirm their vegan or socially responsible status. One brand noted, Velvet Vesper, is known for quality craftsmanship but no direct info about vegans or social responsibility is provided.\n\nI recommend checking brands that emphasize vegan leather made from alternative materials like plant-based or recycled components and have clear commitments to ethical manufacturing and social responsibility. Because the current data doesn\'t list explicit vegan and socially responsible leather jacket brands, you might want to explore ethical fashion retailers or dedicated vegan fashion brands outside this dataset to find a jacket that meets your values.',
│   sources=[
│   │   Source(object_id='db3ab4ed-962a-45f4-8c67-7c5106d6c79f', collection='Ecommerce'),
│   │   Source(object_id='aa8d0abc-1ca9-48a1-b845-078c1c5e3482', collection='Ecommerce'),
│   │   Source(object_id='31be4000-cfa1-4cd3-9aed-f44db62c0288', collection='Brands')
│   ]
)

In [33]:
client.close()